# 2.4 Saving And Loading Projects, Simulations, And Jobs

Production FrequenSolve work should not depend on rerunning the Python authoring script every time a result is inspected or a job is relaunched. The saved project directory is the durable interface: it contains project metadata, simulation contracts, job contracts, logs, and results.

This tutorial builds a small acoustic project, saves the project, saves two jobs, loads each object back from disk, and shows that the loaded job can be submitted through the same site API as a freshly authored job.

By the end, you should be able to reopen a saved project, load an individual simulation or job JSON file, inspect what was persisted, and decide when to resume from disk instead of rebuilding objects from source code.


## How To Read This Tutorial

This notebook is about reproducibility. FrequenSolve projects are not just temporary Python objects; they are durable contracts that can be saved, loaded, inspected, submitted, and analyzed later. That matters for commercial use because production results need to be explainable without re-running the authoring notebook from memory.

Follow the object ownership carefully: a `Project` owns simulations, a simulation owns model/mesh/acquisition/numerics, and a job owns frequencies, workflow, outputs, and result paths. Saving at each level gives you a stable handoff between authoring, execution, and post-processing.

## Persistence Model

FrequenSolve uses three related persistence levels:

| Object | Save API | Load API | What it means |
| --- | --- | --- | --- |
| Project | `project.save()` | `fs.Project.load(project_file_or_dir)` | Restores project metadata and the simulations listed in the project JSON. |
| Simulation | `sim.save()` | `fs.SeismicSimulation.load(sim_file)` | Restores the exported model, mesh, units, BCs, acquisition, and numerics. |
| Job | `job.save()` | `fs.BaseJob.load(job_file)` | Restores the concrete execution request, frequency list, output requests, result path, and linked simulation. |

A project JSON indexes simulations. Jobs are execution requests under the project `jobs/` tree and are loaded by job-file path. This keeps a project from eagerly loading every trial run or QC job, while still making each job reusable.


## Imports

The tutorial uses only the public `import frequensolve as fs` API. No solver is needed for the save/load cells; the final submit cell is strict and requires a configured local solver.


In [ ]:
from pathlib import Path
import json

import numpy as np
import frequensolve as fs

u = fs.ureg


## Author A Small Project

The model is intentionally modest: one acoustic layer, one source, and one receiver line. The important part is not the physics complexity; it is that the project contains the same ingredients as a larger run: model, mesh, boundaries, acquisition, and numerics.


In [ ]:
project = fs.Project(
    name="project",
    pretty_name="Save Load Tutorial",
    path="./scratch/tutorials/save_load_projects_jobs",
    log_level="INFO",
    log_to_console=True,
)

sim = project.new_simulation(
    name="save_load_acoustic",
    physics="acoustic",
    dimension=2,
    units={"length": "km", "velocity": "km/s", "density": "g/cm^3"},
)

model = fs.LayeredModel(name="model", dimension=2, x_limits=[0.0, 1.0])
model.add_surface(name="top", depth=0.0 * u.km)
model.add_layer(
    name="water",
    properties={"Vp": 1.5 * u.km / u.s, "Rho": 1.0 * u.g / u.cm**3},
)
model.add_surface(name="bottom", depth=0.4 * u.km)
sim += model

sim += model.hex_mesh_generator([4, 2])
sim.mesh.set_adapt(elems_per_wave=2.0, order=3, f_low=5.0, f_high=20.0)
sim += fs.BoundaryCondition(conditions=["free"], boundaries=["z_min"])
sim += fs.BoundaryCondition(
    conditions=["pml"],
    boundaries=["x_min", "x_max", "z_max"],
    pml_wavelengths=0.75,
)

acq = fs.Acquisition()
acq.add_sources(kind="scalar", coords=[[0.5, 0.05]])
receiver = fs.ReceiverNode(name="hydrophone")
receiver.add_component(name="p", field="pressure")
acq.add_receiver_group(
    name="surface",
    device=receiver,
    coords=[[x, 0.04] for x in np.linspace(0.1, 0.9, 21)],
)
sim += acq
sim += fs.Discretization()
sim += fs.SolverConfig(tolerance=1.0e-4, grids=3)

model.plot("vp", figsize=(7, 3), aspect="equal")


## Define Jobs Before Saving

Jobs are ordinary Python objects until they are saved or submitted. The time-domain job writes traces. The single-frequency job requests ParaView output for spatial QC. Saving a job also saves its linked simulation so the job JSON points to a concrete simulation file.


In [ ]:
time_job = fs.TimeDomainJob(
    name="time",
    simulation=sim,
    f_min=0.0,
    f_max=20.0,
    T_max=0.5,
)

freq_job = fs.FrequencyDomainJob(
    name="freq_qc",
    simulation=sim,
    f_list=[15.0],
    outputs=[
        fs.VtkOutput.domain(
            name="pv",
            fields=["pressure"],
            properties=["vp", "rho", "Subdomain"],
            show_pml=True,
            upscale=0,
            order=1,
        )
    ],
)


## Save Project, Simulation, And Jobs

The project save returns the project JSON file. The simulation save returns the simulation JSON file. Each job save returns its own job JSON file under `jobs/<simulation>/<job>/`. The helper below displays paths relative to the project root so the notebook remains portable across machines.


In [ ]:
project_file = project.save()
sim_file = sim.save()
time_job_file = time_job.save()
freq_job_file = freq_job.save()

project_root = project.path

def rel(path):
    return str(Path(path).resolve().relative_to(project_root))

{
    "project": rel(project_file),
    "simulation": rel(sim_file),
    "time_job": rel(time_job_file),
    "frequency_job": rel(freq_job_file),
}


## Inspect The Saved Layout

The saved files are the handoff between authoring, execution, and later analysis. Inspecting them is a production habit: it confirms the job is not only present in memory but also recoverable from disk.


In [ ]:
saved_files = sorted(
    path.relative_to(project_root)
    for path in project_root.rglob("*.json")
    if ".tmp" not in path.name
)
[str(path) for path in saved_files]


## Load The Project And Simulation

`Project.load(...)` can receive either the project JSON file or a directory containing one project JSON file. The loaded project restores its listed simulations, and direct simulation loading is useful when you want to inspect or reuse one simulation contract without loading the whole project.


In [ ]:
loaded_project = fs.Project.load(project_file)
loaded_project_from_dir = fs.Project.load(project_root)
loaded_sim_from_project = loaded_project.simulations["save_load_acoustic"]
loaded_sim_direct = fs.SeismicSimulation.load(sim_file)

{
    "project_name": loaded_project.name,
    "project_from_dir": loaded_project_from_dir.name,
    "simulation_from_project": loaded_sim_from_project.name,
    "simulation_direct": loaded_sim_direct.name,
    "loaded_physics": loaded_sim_direct.physics,
    "loaded_receiver_groups": [group.name for group in loaded_sim_direct.acquisition.receiver_groups],
}


## Load Saved Jobs

Jobs are loaded by job JSON path. A loaded job includes its linked simulation, output requests, frequency list, and result path. This is the workflow to use in a post-processing notebook, rerun notebook, or site-specific launch notebook when the authoring step already happened earlier.


In [ ]:
loaded_time_job = fs.BaseJob.load(time_job_file)
loaded_freq_job = fs.BaseJob.load(freq_job_file)

{
    "time_job_class": type(loaded_time_job).__name__,
    "time_job_name": loaded_time_job.name,
    "time_job_simulation": loaded_time_job.simulation.name,
    "time_job_frequencies": len(loaded_time_job.f_list),
    "freq_job_class": type(loaded_freq_job).__name__,
    "freq_job_name": loaded_freq_job.name,
    "freq_job_outputs": [output.name for output in loaded_freq_job.outputs.vtk],
}


## Inspect A Job Contract

The job JSON is what a site stages or submits. It records the linked simulation, workflow, frequency list, output requests, and result path. Inspecting it before a remote run helps catch mistakes such as a missing ParaView output, a wrong frequency list, or an unexpected simulation path.


In [ ]:
job_payload = json.loads(Path(freq_job_file).read_text())
{
    "schema": job_payload["schema"],
    "type": job_payload["_type"],
    "simulation": job_payload["simulation"],
    "workflow": job_payload["workflow"],
    "n_frequencies": len(job_payload["f_list"]),
    "outputs": list(job_payload["Outputs"].keys()),
    "result_path": job_payload["result_path"],
}


## Submit A Loaded Job

The loaded job is the concrete saved `BaseJob` subclass, not a lightweight stub. Submitting it uses the same strict site workflow as a freshly authored job. This cell requires a configured local solver; if the solver is unavailable or fails, the notebook should stop here with the saved job and logs still inspectable.


In [ ]:
site = fs.Site()
result = site.submit(loaded_time_job).wait()
traces = result.traces(upscale=4)
traces.summary


## Before Moving On

The practical test is whether another notebook can load the saved project or job and submit it without reconstructing the model in code. If that works, the project directory has become the source of truth.

Use this pattern for handoffs: author and save once, load and submit from site-specific notebooks, then load again for analysis. That keeps modeling decisions, infrastructure decisions, and interpretation decisions separated.

## Result Review Checklist

Persistence is successful when the saved files are reusable without the original authoring cells.

| Artifact | What it proves |
| --- | --- |
| Project JSON | The project can be reopened and restores its listed simulations. |
| Simulation JSON | Model, mesh, units, BCs, acquisition, and numerics can be loaded directly. |
| Job JSON | Frequency list, outputs, result path, and linked simulation are recoverable. |
| Loaded job submission | A loaded job can be passed to a site exactly like a newly constructed job. |
| Relative path display | The tutorial output is portable and does not depend on a developer machine path. |

For production workflows, keep project directories stable, save jobs before submission, and use descriptive job names that encode the purpose of the run. That makes later analysis notebooks independent of the original authoring script.
